In [10]:
# COLAB SETUP

!rm -rf /content/proto-tsrl
!git clone https://github.com/haiyan-wang/proto-tsrl.git /content/proto-tsrl
%cd /content/proto-tsrl

from google.colab import drive
drive.mount('/content/drive')

import sys
import os

project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(project_root)

Cloning into '/content/proto-tsrl'...
remote: Enumerating objects: 72, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 72 (delta 16), reused 62 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (72/72), 861.85 KiB | 4.90 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/proto-tsrl


In [ ]:
import functools 

import numpy as np

from sklearn.model_selection import StratifiedShuffleSplit

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F

from src.experiments.ppg.ppg_model import PPGModel
from src.utils.sampling_utils import *
from src.utils.training_utils import *

In [ ]:
# SETTINGS

SEED = 42

# device
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# data quality 
INCLUDE_CLEAN_DATA = True
INCLUDE_SEMINOISY_DATA = True
INCLUDE_NOISY_DATA = False

# dataloaders 
BATCH_SIZE = 256
VAL_RATIO = 0.05 
N_WORKERS = 4

# contrastive sampling 
COLLATE_FN_KWARGS = {
    'min_len': 100,
    'max_len': 300,
    'num_neg': 5,
    'min_overlap': 0.3,
    'min_var': 0.5,
    'max_var': 2.0
}

# architecture
MASK_PROB = 0.2
MODEL = PPGModel(representation_dimension = 10, mask_probability = MASK_PROB)
if torch.cuda.device_count() > 1:
    MODEL = nn.DataParallel(MODEL)
elif not isinstance(MODEL, nn.DataParallel):
    MODEL = nn.DataParallel(MODEL)

# model settings (all defaults)
MID_WEIGHT = 0.5,
PROTO_NEG_MARGIN = 0.1,
PROTO_DIVERSITY_THRESHOLD = 0.2,
LAMBDA_PROTO = 1.0,
TEMPERATURE = 1.0,
LAMBDA_REPR = 1.0,
GRADIENT_CLIP = None

# schedule
N_EPOCHS_STAGE1 = 6
N_EPOCHS_STAGE2 = 6
N_EPOCHS_STAGE3 = 3
TOTAL_EPOCHS = N_EPOCHS_STAGE1 + N_EPOCHS_STAGE2 + N_EPOCHS_STAGE3

# logging
CHECKPOINT_EPOCHS = [*range(1, TOTAL_EPOCHS + 1)] # 1-indexed 
SAVE_DIR = "/content/drive/MyDrive/Duke/Senior Year/Thesis/experiments/ppg"

# optimizer
OPTIMIZERS = {}

SCHEDULER = {}

In [ ]:
# DATA LOADING FUNCTIONS 

# quality: high - [0,0,1] med - [0,1,0] low - [1,0,0]
# afib: pos - [0,1]
def load_data(
        clean : bool, 
        seminoisy : bool, 
        noisy : bool, 
        dataset : str, 
        return_labels : bool = True
    ):
    """
    Load PPG data from Drive 

    Arguments
    ---------
        - clean, seminoisy, noisy: bools indicating whether to include data of each quality level
        - dataset: 'train' or 'test'
        - return_labels: whether to return labels (if False, only returns signals)

    Returns
    -------
        - X: (N, L) ndarray of PPG signals
        - y: (N,) ndarray of binary labels indicating presence of afib if return_labels = True
    """

    file_path = '/content/drive/MyDrive/Duke/Senior Year/Thesis/ppg_data/'
    with np.load(file_path + f'{dataset}.npz') as data:
        ppg_signal = data['signal']
        ppg_qual = data['qa_label']
        rhythm = data['rhythm']

    qual_mask = np.zeros(ppg_qual.shape[0], dtype = bool)
    if clean:
        qual_mask = (qual_mask | np.all(ppg_qual == np.array([0, 0, 1]), axis = 1))
    if seminoisy:
        qual_mask = (qual_mask | np.all(ppg_qual == np.array([0, 1, 0]), axis = 1))
    if noisy:
        qual_mask = (qual_mask | np.all(ppg_qual == np.array([1, 0, 0]), axis = 1))

    X, y = ppg_signal[qual_mask], rhythm[qual_mask]

    afib_label = np.array([0, 1])
    y = np.all(y == afib_label, axis = 1)

    if return_labels:
        return X, y
    
    return X

def make_dataloaders(
        X : np.ndarray, 
        y : np.ndarray = None, 
        batch_size : int = 256, 
        val_ratio : float = 0.05, 
        seed : int = 42, 
        num_workers : int = 4,
        collage_fn_kwargs : dict = None
    ) -> tuple[DataLoader, DataLoader]:
    """
    Create training and validation dataloaders from PPG signals and labels. If labels exist, they can be passed for use in stratified splitting.

    IMPORTANT: TimeSeriesDataset class takes signals as list of tensors of shape [T_i, F]

    Arguments
    ---------
        - X: (N, L) ndarray of PPG signals
        - y: (N,) ndarray of binary labels
        - batch_size: batch size for dataloaders
        - val_ratio: proportion of data to use for validation
        - seed: random seed for reproducibility
        - num_workers: number of worker processes for data loading
        - collage_fn_kwargs: dict of kwargs to pass to contrastive_collate function (if None, defaults will be used)

    Returns
    -------
        - dl_train: DataLoader for training data
        - dl_val: DataLoader for validation data
    """

    if y:
        sss = StratifiedShuffleSplit(n_splits = 1, test_size = val_ratio, random_state = seed)
        train_idx, val_idx = next(sss.split(X, y))
    else: 
        n = X.shape[0]
        indices = np.random.default_rng(seed).permutation(n)
        n_val = int(n * val_ratio)
        train_idx = indices[n_val:]
        val_idx = indices[:n_val]

    sig_train = X[train_idx].astype(np.float32)
    sig_val = X[val_idx].astype(np.float32)
    ds_train = TimeSeriesDataset(sig_train.tolist())
    ds_val = TimeSeriesDataset(sig_val.tolist())

    min_len = collage_fn_kwargs.get('min_len', X.shape[1]) if collage_fn_kwargs else X.shape[1]
    max_len = collage_fn_kwargs.get('max_len', X.shape[1]) if collage_fn_kwargs else X.shape[1]
    num_neg = collage_fn_kwargs.get('num_neg', 5) if collage_fn_kwargs else 4
    min_overlap = collage_fn_kwargs.get('min_overlap', 0.3) if collage_fn_kwargs else 0.3
    min_var = collage_fn_kwargs.get('min_var', 0.5) if collage_fn_kwargs else 0.5
    max_var = collage_fn_kwargs.get('max_var', 2) if collage_fn_kwargs else 2

    collate_fn = functools.partial(
        contrastive_collate,
        min_len = min_len,
        max_len = max_len,
        num_neg = num_neg,
        min_overlap = min_overlap,
        min_var = min_var,
        max_var = max_var
    )

    dl_train = DataLoader(
        ds_train,
        batch_size = batch_size,
        shuffle = True,
        collate_fn = collate_fn,
        num_workers = num_workers,
        pin_memory = True,
        drop_last = True
    )
    
    dl_val = DataLoader(
        ds_val,
        batch_size = batch_size,
        shuffle = False,
        collate_fn = collate_fn,
        num_workers = num_workers,
        pin_memory = True,
        drop_last = True
    )

    return dl_train, dl_val

In [ ]:
# LOAD DATA 

X_train, y_train = load_data(
    clean = INCLUDE_CLEAN_DATA,
    seminoisy = INCLUDE_SEMINOISY_DATA,
    noisy = INCLUDE_NOISY_DATA,
    dataset = 'train',
    return_labels = False
)

X_test, y_test = load_data(
    clean = INCLUDE_CLEAN_DATA,
    seminoisy = INCLUDE_SEMINOISY_DATA,
    noisy = INCLUDE_NOISY_DATA,
    dataset = 'test',
    return_labels = True
)

print(f'X_train shape: {X_train.shape}')
print(f'Train set positive samples: {np.sum(y_train)}')
print(f'X_test shape: {X_test.shape}')
print(f'Test set positive samples: {np.sum(y_test)}')

dl_train, dl_val = make_dataloaders(
    X_train, 
    y_train,
    batch_size = BATCH_SIZE, 
    val_ratio = VAL_RATIO, 
    seed = SEED, 
    num_workers = N_WORKERS,
    collage_fn_kwargs = COLLATE_FN_KWARGS
)

In [ ]:
### TRAINING 

history = run_training(
    model = MODEL,
    train_loader = dl_train,
    val_loader = dl_val,
    device = DEVICE,
    optimizer_dict = OPTIMIZERS,
    epochs_stage1 = N_EPOCHS_STAGE1,
    epochs_stage2 = N_EPOCHS_STAGE2,
    epochs_stage3 = N_EPOCHS_STAGE3,
    scheduler_dict = SCHEDULER,
    mid_weight = MID_WEIGHT,
    proto_neg_margin = PROTO_NEG_MARGIN,
    proto_diversity_threshold = PROTO_DIVERSITY_THRESHOLD,
    lambda_proto = LAMBDA_PROTO,
    temperature = TEMPERATURE,
    lambda_repr = LAMBDA_REPR,
    grad_clip_norm = GRADIENT_CLIP,
    checkpoint_path = '{SAVE_DIR}',
    checkpoint_epochs = CHECKPOINT_EPOCHS,
    collector_fn = None,
    use_amp = True
)